In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Standard Drive Mount
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    raw_dir = Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/raw_mega")
    out_dir = Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed_mega")
    out_dir.mkdir(parents=True, exist_ok=True)
except:
    raw_dir = Path("../data/raw_mega")
    out_dir = Path("../data/processed_mega")

print(f"Reading from: {raw_dir}")

In [ ]:
def process_month(year, month):
    bid_path = raw_dir / f"EURUSD_{year}_{month:02d}_bid.parquet"
    ask_path = raw_dir / f"EURUSD_{year}_{month:02d}_ask.parquet"
    
    if not bid_path.exists() or not ask_path.exists():
        return None
        
    # Read core columns only to save RAM
    b = pd.read_parquet(bid_path, columns=['datetime', 'price']).rename(columns={'price': 'bid'})
    a = pd.read_parquet(ask_path, columns=['datetime', 'price']).rename(columns={'price': 'ask'})
    
    # Merge on Ticks (merge_asof is the industry standard for HFT)
    df = pd.merge_asof(b, a, on='datetime', direction='backward')
    
    # Calculate Mid-price and Log Returns
    df['mid'] = (df['bid'] + df['ask']) / 2.0
    df['returns'] = np.log(df['mid'] / df['mid'].shift(1))
    
    # CLEANING: Drop holidays/weekends where vol is 0
    df = df[df['returns'] != 0].dropna()
    
    # HARD DOWNSAMPLING (The 'Mega' Scale trick)
    # We pre-average every 100 ticks to turn the noise into signal
    # and reduce the row count from 50M to 500k
    df = df.rolling(window=100).mean().iloc[::100, :].dropna()
    
    return df[['datetime', 'returns']]

# Assemble the Fleet
mega_data = []
for y in range(2021, 2027):
    for m in range(1, 13):
        print(f"Cleaning {y}-{m:02d}...")
        df_month = process_month(y, m)
        if df_month is not None:
            mega_data.append(df_month)

# Combine into one Master Dataframe
master_df = pd.concat(mega_data, ignore_index=True)
master_path = out_dir / "eurusd_mega_master_21_26.parquet"
master_df.to_parquet(master_path)

print(f"🏁 MASTER DATASET SAVED: {len(master_df)} rows at {master_path}")

In [ ]:
def process_month(year, month):
    bid_path = raw_dir / f"EURUSD_{year}_{month:02d}_bid.parquet"
    ask_path = raw_dir / f"EURUSD_{year}_{month:02d}_ask.parquet"
    
    if not bid_path.exists() or not ask_path.exists():
        return None
        
    # Read core columns only to save RAM
    b = pd.read_parquet(bid_path, columns=['datetime', 'price']).rename(columns={'price': 'bid'})
    a = pd.read_parquet(ask_path, columns=['datetime', 'price']).rename(columns={'price': 'ask'})
    
    # Merge on Ticks (merge_asof is the industry standard for HFT)
    df = pd.merge_asof(b, a, on='datetime', direction='backward')
    
    # Calculate Mid-price and Log Returns
    df['mid'] = (df['bid'] + df['ask']) / 2.0
    df['returns'] = np.log(df['mid'] / df['mid'].shift(1))
    
    # CLEANING: Drop holidays/weekends where vol is 0
    df = df[df['returns'] != 0].dropna()
    
    # HARD DOWNSAMPLING (The 'Mega' Scale trick)
    # We pre-average every 100 ticks to turn the noise into signal
    # and reduce the row count from 50M to 500k
    df = df.rolling(window=100).mean().iloc[::100, :].dropna()
    
    return df[['datetime', 'returns']]

# Assemble the Fleet
mega_data = []
for y in range(2021, 2027):
    for m in range(1, 13):
        print(f"Cleaning {y}-{m:02d}...")
        df_month = process_month(y, m)
        if df_month is not None:
            mega_data.append(df_month)

# Combine into one Master Dataframe
master_df = pd.concat(mega_data, ignore_index=True)
master_path = out_dir / "eurusd_mega_master_21_26.parquet"
master_df.to_parquet(master_path)

print(f"🏁 MASTER DATASET SAVED: {len(master_df)} rows at {master_path}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))
plt.plot(master_df['datetime'], master_df['returns'].rolling(100).std(), color='navy', lw=0.5)
plt.title("5-Year Volatility Regime Map (2021-2026)")
plt.ylabel("Rolling Volatility")
plt.grid(alpha=0.3)
plt.show()